In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.preprocessing import image
from google.colab import drive
import numpy as np
import os

drive.mount('/content/drive')

dataset_path = "/content/drive/MyDrive/Neuro/Dataset/"

print(os.listdir(dataset_path))

train_data = tf.keras.preprocessing.image_dataset_from_directory(
    dataset_path + "Train",
    image_size=(224,224),
    batch_size=32
)

test_data = tf.keras.preprocessing.image_dataset_from_directory(
    dataset_path + "Test",
    image_size=(224,224),
    batch_size=32
)

AUTOTUNE = tf.data.AUTOTUNE
train_data = train_data.prefetch(buffer_size=AUTOTUNE)
test_data = test_data.prefetch(buffer_size=AUTOTUNE)

data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1)
])

model = models.Sequential([

    layers.Input(shape=(224,224,3)),

    data_augmentation,

    layers.Rescaling(1./255),

    layers.Conv2D(32,(3,3),activation='relu'),
    layers.MaxPooling2D(2,2),

    layers.Conv2D(64,(3,3),activation='relu'),
    layers.MaxPooling2D(2,2),

    layers.Conv2D(128,(3,3),activation='relu'),
    layers.MaxPooling2D(2,2),

    layers.Conv2D(256,(3,3),activation='relu'),
    layers.MaxPooling2D(2,2),

    layers.Flatten(),

    layers.Dense(256,activation='relu'),

    layers.Dropout(0.5),

    layers.Dense(1,activation='sigmoid')
])

model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model.summary()

history = model.fit(
    train_data,
    validation_data=test_data,
    epochs=20
)

test_loss, test_acc = model.evaluate(test_data)

print("Test Accuracy:", test_acc*100, "%")

img = image.load_img(
    "/content/drive/MyDrive/Neuro/Dataset/Test/YES/Y1.JPG",
    target_size=(224,224)
)

img_array = image.img_to_array(img)/255.0
img_array = np.expand_dims(img_array, axis=0)

prediction = model.predict(img_array)

print("Tumor Image Prediction:", prediction[0][0])

if prediction[0][0] > 0.5:
    print("Disease Detected (Brain Tumor)")
else:
    print("Normal Brain")

img = image.load_img(
    "/content/drive/MyDrive/Neuro/Dataset/Test/NO/N1.JPG",
    target_size=(224,224)
)

img_array = image.img_to_array(img)/255.0
img_array = np.expand_dims(img_array, axis=0)

prediction = model.predict(img_array)

print("Normal Image Prediction:", prediction[0][0])

if prediction[0][0] > 0.5:
    print("Disease Detected (Brain Tumor)")
else:
    print("Normal Brain")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
['Train', 'Test']
Found 178 files belonging to 2 classes.
Found 75 files belonging to 2 classes.


Model: "sequential_5"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ sequential_4 (Sequential)       │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ rescaling_2 (Rescaling)         │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_4 (Conv2D)               │ (None, 222, 222, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_4 (MaxPooling2D)  │ (None, 111, 111, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_5 (Conv2D)               │ (None, 109, 109, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_5 (MaxPooling2D)  │ (None, 54, 54, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_6 (Conv2D)               │ (None, 52, 52, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_6 (MaxPooling2D)  │ (None, 26, 26, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_7 (Conv2D)               │ (None, 24, 24, 256)    │       295,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_7 (MaxPooling2D)  │ (None, 12, 12, 256)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_1 (Flatten)             │ (None, 36864)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 256)            │     9,437,440 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 1)              │           257 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 9,826,113 (37.48 MB)

 Trainable params: 9,826,113 (37.48 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/20
6/6 ━━━━━━━━━━━━━━━━━━━━ 30s 4s/step - accuracy: 0.5031 - loss: 0.9798 - val_accuracy: 0.6000 - val_loss: 0.6321
Epoch 2/20
6/6 ━━━━━━━━━━━━━━━━━━━━ 27s 5s/step - accuracy: 0.6769 - loss: 0.6312 - val_accuracy: 0.6800 - val_loss: 0.5717
Epoch 3/20
6/6 ━━━━━━━━━━━━━━━━━━━━ 27s 5s/step - accuracy: 0.7108 - loss: 0.6079 - val_accuracy: 0.7600 - val_loss: 0.4816
Epoch 4/20
6/6 ━━━━━━━━━━━━━━━━━━━━ 25s 4s/step - accuracy: 0.7191 - loss: 0.6014 - val_accuracy: 0.7867 - val_loss: 0.4957
Epoch 5/20
6/6 ━━━━━━━━━━━━━━━━━━━━ 41s 4s/step - accuracy: 0.7153 - loss: 0.5890 - val_accuracy: 0.8267 - val_loss: 0.5693
Epoch 6/20
6/6 ━━━━━━━━━━━━━━━━━━━━ 41s 4s/step - accuracy: 0.7662 - loss: 0.5857 - val_accuracy: 0.8800 - val_loss: 0.4468
Epoch 7/20
6/6 ━━━━━━━━━━━━━━━━━━━━ 43s 5s/step - accuracy: 0.7473 - loss: 0.5398 - val_accuracy: 0.8800 - val_loss: 0.3984
Epoch 8/20
6/6 ━━━━━━━━━━━━━━━━━━━━ 39s 4s/step - accuracy: 0.7800 - loss: 0.5123 - val_accuracy: 0.8800 - val_loss: 0.4009
Epoch 9/